In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Fix complet versions compatibles
!pip uninstall transformers tokenizers -y -q
!pip install tokenizers==0.20.3 -q
!pip install transformers==4.46.1 --no-deps -q
!pip install peft --no-deps -q
!pip install accelerate -q
!pip install bitsandbytes>=0.46.1 -q
!pip install Pillow -q

print("✅ Terminé ! Redémarre le runtime !")
print("→ Exécution → Redémarrer la session")

✅ Terminé ! Redémarre le runtime !
→ Exécution → Redémarrer la session


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from collections import Counter

BASE_PATH  = '/content/drive/MyDrive/LLM/RAF-CE'
IMAGE_PATH = BASE_PATH + '/Image/aligned'
LABEL_PATH = BASE_PATH + '/EmoLabel/list_patition_label.txt'
CHECKPOINT = '/content/drive/MyDrive/LLM/llava_rafce_epoch4'
SAVE_PATH  = '/content/drive/MyDrive/LLM/'

emotions = {
    1: "Happily Surprised",
    2: "Happily Disgusted",
    3: "Sadly Fearful",
    4: "Sadly Angry",
    5: "Sadly Surprised",
    6: "Sadly Disgusted",
    7: "Fearfully Angry",
    8: "Fearfully Surprised",
    9: "Angrily Surprised",
    10: "Angrily Disgusted",
    11: "Disgustedly Surprised"
}

train_data, test_data = [], []
with open(LABEL_PATH, 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) == 2:
            name  = parts[0]
            label = int(parts[1])
            if name.startswith('train'):
                train_data.append((name, label))
            else:
                test_data.append((name, label))

print(f"✅ Train : {len(train_data)}")
print(f"✅ Test  : {len(test_data)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Train : 3162
✅ Test  : 792


In [ ]:
#cellule 4 LLaVA nécessite un token HuggingFace
# pour télécharger le modèle
# Créer compte sur huggingface.co
# → Settings → Tokens → New Token

from huggingface_hub import login

# Mets ton token ici
HF_TOKEN = "your token"  # ← ton token
login(token=HF_TOKEN)

print("✅ HuggingFace connecté !")

✅ HuggingFace connecté !


In [ ]:
#cellule 5 Charger LLaVA-7B avec quantization 4bit
# 4bit = réduit la RAM de 15GB → 6GB
# Permet de tourner sur Colab gratuit !

import torch
from transformers import (
    LlavaForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig)

# Configuration 4bit pour économiser RAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16)

MODEL_ID = "llava-hf/llava-1.5-7b-hf"

print("🔄 Chargement LLaVA-7B...")
print("⏳ ~3-5 minutes...")

model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN)

print("✅ LLaVA-7B chargé !")

🔄 Chargement LLaVA-7B...
⏳ ~3-5 minutes...


config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/41.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

Some kwargs in processor config are unused and will not have any effect: num_additional_image_tokens. 


✅ LLaVA-7B chargé !


In [ ]:
from peft import (
    PeftModel,
    prepare_model_for_kbit_training)

model = prepare_model_for_kbit_training(model)
model = PeftModel.from_pretrained(
    model,
    CHECKPOINT,
    is_trainable=True)

print("✅ Checkpoint epoch 4 chargé !")
free = torch.cuda.mem_get_info()[0]/1e9
used = 15.6 - free
print(f"💾 RAM : {used:.1f}/15.6 GB")

✅ Checkpoint epoch 4 chargé !
💾 RAM : 5.8/15.6 GB


In [ ]:
import os
emotion_aus = {
    1:  ("Happily Surprised",
         "raised eyebrows (AU1, AU2) and a wide smile (AU12, AU25)"),
    2:  ("Happily Disgusted",
         "smile (AU12) combined with nose wrinkle (AU9) and upper lip raise (AU10)"),
    3:  ("Sadly Fearful",
         "inner brow raise (AU1), lip corner depression (AU15) and eye widening (AU5)"),
    4:  ("Sadly Angry",
         "brow lowering (AU4), lip corner depression (AU15) and lid tightening (AU23)"),
    5:  ("Sadly Surprised",
         "inner brow raise (AU1), lip corner depression (AU15) and jaw drop (AU27)"),
    6:  ("Sadly Disgusted",
         "nose wrinkle (AU9), lip corner depression (AU15) and brow lowering (AU4)"),
    7:  ("Fearfully Angry",
         "brow lowering (AU4), eye widening (AU5) and lip tightening (AU23)"),
    8:  ("Fearfully Surprised",
         "inner brow raise (AU1), eye widening (AU5) and jaw drop (AU27)"),
    9:  ("Angrily Surprised",
         "brow lowering (AU4), eye widening (AU5) and jaw drop (AU27)"),
    10: ("Angrily Disgusted",
         "brow lowering (AU4), nose wrinkle (AU9) and lip corner pull (AU14)"),
    11: ("Disgustedly Surprised",
         "nose wrinkle (AU9), jaw drop (AU27) and eye widening (AU5)")
}

llava_train, llava_test = [], []
for name, label in train_data:
    aligned  = name.replace('.jpg', '_aligned.jpg')
    img_path = os.path.join(IMAGE_PATH, aligned)
    if os.path.exists(img_path):
        emotion_name, aus_desc = emotion_aus[label]
        llava_train.append({
            "image_path": img_path,
            "label"     : label,
            "question"  : "What emotion does this person express? Describe the facial action units.",
            "answer"    : f"This person expresses {emotion_name}. The facial cues include {aus_desc}."
        })

for name, label in test_data:
    aligned  = name.replace('.jpg', '_aligned.jpg')
    img_path = os.path.join(IMAGE_PATH, aligned)
    if os.path.exists(img_path):
        emotion_name, aus_desc = emotion_aus[label]
        llava_test.append({
            "image_path": img_path,
            "label"     : label,
            "question"  : "What emotion does this person express? Describe the facial action units.",
            "answer"    : f"This person expresses {emotion_name}. The facial cues include {aus_desc}."
        })

print(f"✅ Train : {len(llava_train)}")
print(f"✅ Test  : {len(llava_test)}")

✅ Train : 3162
✅ Test  : 792


In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class LLaVADataset(Dataset):
    def __init__(self, data, processor):
        self.data      = data
        self.processor = processor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item  = self.data[idx]
        image = Image.open(
            item['image_path']).convert('RGB')
        prompt = (
            f"USER: <image>\n"
            f"{item['question']}\n"
            f"ASSISTANT: {item['answer']}")
        inputs = self.processor(
            text=prompt,
            images=image,
            return_tensors="pt",
            padding="max_length",
            truncation=False,
            max_length=1024)
        return {k: v.squeeze(0)
                for k, v in inputs.items()}

train_dataset_llava = LLaVADataset(
    llava_train, processor)
test_dataset_llava  = LLaVADataset(
    llava_test, processor)

train_loader_llava = DataLoader(
    train_dataset_llava,
    batch_size=2, shuffle=True)
test_loader_llava  = DataLoader(
    test_dataset_llava,
    batch_size=2, shuffle=False)

print(f"✅ Train : {len(train_dataset_llava)}")
print(f"✅ Test  : {len(test_dataset_llava)}")

✅ Train : 3162
✅ Test  : 792


In [ ]:
import random
random.seed(42)

subset_size = int(len(llava_train) * 0.5)
llava_train_small = random.sample(
    llava_train, subset_size)

train_dataset_llava = LLaVADataset(
    llava_train_small, processor)
train_loader_llava  = DataLoader(
    train_dataset_llava,
    batch_size=2, shuffle=True)

print(f"✅ Subset 50% : {len(llava_train_small)}")
print(f"✅ Batches    : {len(train_loader_llava)}")

✅ Subset 50% : 1581
✅ Batches    : 791


In [ ]:
from torch.amp import autocast, GradScaler
from torch.optim import AdamW

device    = torch.device('cuda')
optimizer = AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01)
scaler = GradScaler('cuda')

START_EPOCH = 4
NUM_EPOCHS  = 5
MAX_STEPS   = 791

print("🚀 Reprise depuis Epoch 5 !")
print(f"   Epochs : {NUM_EPOCHS}")
print(f"   Steps  : {MAX_STEPS}/epoch")
print("="*50)
print(f"   Epochs : {NUM_EPOCHS}")
print(f"   Steps  : {MAX_STEPS}/epoch")
print("="*50)

model.train()
for epoch in range(START_EPOCH, NUM_EPOCHS):
    total_loss = 0.0
    steps      = 0

    for batch in train_loader_llava:
        if steps >= MAX_STEPS:
            break

        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        pixel_values   = batch['pixel_values'].to(device)

        optimizer.zero_grad()
        with autocast('cuda'):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                labels=input_ids)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        steps      += 1

        if steps % 100 == 0:
            avg = total_loss / steps
            print(f"  Epoch {epoch+1} "
                  f"Step {steps:4d} "
                  f"Loss: {avg:.4f}")

    avg_loss = total_loss / steps
    print(f"\n✅ Epoch [{epoch+1}/{NUM_EPOCHS}]"
          f" Loss: {avg_loss:.4f}")

    model.save_pretrained(
        f'{SAVE_PATH}llava_rafce_epoch{epoch+1}')
    print(f"💾 Epoch {epoch+1} sauvegardée !")

print("\n🎉 Fine-tuning terminé !")

🚀 Reprise depuis Epoch 5 !
   Epochs : 5
   Steps  : 791/epoch
   Epochs : 5
   Steps  : 791/epoch


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


  Epoch 5 Step  100 Loss: 3.4507
  Epoch 5 Step  200 Loss: 3.4503
  Epoch 5 Step  300 Loss: 3.4502
  Epoch 5 Step  400 Loss: 3.4499
  Epoch 5 Step  500 Loss: 3.4497
  Epoch 5 Step  600 Loss: 3.4495
  Epoch 5 Step  700 Loss: 3.4492

✅ Epoch [5/5] Loss: 3.4490


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


💾 Epoch 5 sauvegardée !

🎉 Fine-tuning terminé !


In [ ]:
# ============================================
# Cellule 6 — Préparation Données Test RAF-CE
# ============================================

import os

emotion_aus = {
    1:  ("Happily Surprised",
         "raised eyebrows (AU1, AU2) and a wide smile (AU12, AU25)"),
    2:  ("Happily Disgusted",
         "smile (AU12) combined with nose wrinkle (AU9) and upper lip raise (AU10)"),
    3:  ("Sadly Fearful",
         "inner brow raise (AU1), lip corner depression (AU15) and eye widening (AU5)"),
    4:  ("Sadly Angry",
         "brow lowering (AU4), lip corner depression (AU15) and lid tightening (AU23)"),
    5:  ("Sadly Surprised",
         "inner brow raise (AU1), lip corner depression (AU15) and jaw drop (AU27)"),
    6:  ("Sadly Disgusted",
         "nose wrinkle (AU9), lip corner depression (AU15) and brow lowering (AU4)"),
    7:  ("Fearfully Angry",
         "brow lowering (AU4), eye widening (AU5) and lip tightening (AU23)"),
    8:  ("Fearfully Surprised",
         "inner brow raise (AU1), eye widening (AU5) and jaw drop (AU27)"),
    9:  ("Angrily Surprised",
         "brow lowering (AU4), eye widening (AU5) and jaw drop (AU27)"),
    10: ("Angrily Disgusted",
         "brow lowering (AU4), nose wrinkle (AU9) and lip corner pull (AU14)"),
    11: ("Disgustedly Surprised",
         "nose wrinkle (AU9), jaw drop (AU27) and eye widening (AU5)")
}

# Préparer uniquement les données test
llava_test = []
for name, label in test_data:
    aligned  = name.replace('.jpg', '_aligned.jpg')
    img_path = os.path.join(IMAGE_PATH, aligned)
    if os.path.exists(img_path):
        emotion_name, aus_desc = emotion_aus[label]
        llava_test.append({
            "image_path": img_path,
            "label"     : label,
            "question"  : "What emotion does this person express? Describe the facial action units.",
            "answer"    : f"This person expresses {emotion_name}. The facial cues include {aus_desc}."
        })

print(f"✅ Test : {len(llava_test)} images prêtes pour l'évaluation")

✅ Test : 792 images prêtes pour l'évaluation


In [ ]:
# ============================================================
# Cellule 7 — Évaluation LLaVA Epoch 3 et Epoch 5
# Fix : dictionnaire étendu + extraction ASSISTANT
# ============================================================

import torch
import gc
from sklearn.metrics import f1_score, accuracy_score
from PIL import Image
from tqdm import tqdm
from peft import PeftModel

device = torch.device('cuda')

# Dictionnaire étendu avec toutes les réponses LLaVA
name_to_label = {
    # Noms complets
    "Happily Surprised"    : 1,
    "Happily Disgusted"    : 2,
    "Sadly Fearful"        : 3,
    "Sadly Angry"          : 4,
    "Sadly Surprised"      : 5,
    "Sadly Disgusted"      : 6,
    "Fearfully Angry"      : 7,
    "Fearfully Surprised"  : 8,
    "Angrily Surprised"    : 9,
    "Angrily Disgusted"    : 10,
    "Disgustedly Surprised": 11,
    # Réponses courtes observées
    "Sad"        : 6,
    "Sadly"      : 6,
    "Fear"       : 3,
    "Happiness"  : 1,
    "Angrily Sad": 4,
}

# Trier par longueur décroissante
# pour éviter les matchs partiels
sorted_emotions = sorted(
    name_to_label.items(),
    key=lambda x: len(x[0]),
    reverse=True)

# ============================================================
# Fonction d'évaluation
# ============================================================
def evaluate_checkpoint(
    model, processor,
    test_data, epoch_name):

    print(f"\n{'='*50}")
    print(f"🔍 Évaluation {epoch_name}")
    print(f"{'='*50}")

    model.eval()
    all_preds  = []
    all_labels = []
    errors     = 0

    for item in tqdm(llava_test,
                     desc=epoch_name):
        try:
            # Charger image
            image = Image.open(
                item['image_path']
            ).convert('RGB')

            # Prompt LLaVA
            prompt = (
                "USER: <image>\n"
                "What emotion does this "
                "person express? Answer "
                "with only the emotion name.\n"
                "ASSISTANT:")

            inputs = processor(
                text=prompt,
                images=image,
                return_tensors="pt"
            ).to(device)

            # Inférence
            with torch.no_grad():
                output = model.generate(
                    **inputs,
                    max_new_tokens=20,
                    do_sample=False)

            # Décoder et extraire après ASSISTANT:
            response = processor.decode(
                output[0],
                skip_special_tokens=True)

            if "ASSISTANT:" in response:
                response = response.split(
                    "ASSISTANT:")[-1].strip()

            # Chercher émotion du plus long au plus court
            predicted_label = None
            for name, label in sorted_emotions:
                if name.lower() in response.lower():
                    predicted_label = label
                    break

            # Si rien trouvé → erreur
            if predicted_label is None:
                predicted_label = 1
                errors += 1

            all_preds.append(predicted_label - 1)
            all_labels.append(item['label'] - 1)

        except:
            errors += 1
            continue

    # Métriques
    acc = accuracy_score(
        all_labels, all_preds) * 100
    f1  = f1_score(
        all_labels, all_preds,
        average='macro') * 100

    print(f"\n📊 {epoch_name} :")
    print(f"   Accuracy : {acc:.2f}%")
    print(f"   F1 Macro : {f1:.2f}%")
    print(f"   Erreurs  : {errors}/100")

    return acc, f1

# ============================================================
# Évaluation Epoch 3
# ============================================================
print("🔄 Chargement Epoch 3...")
model_eval = PeftModel.from_pretrained(
    model,
    '/content/drive/MyDrive/LLM/llava_rafce_epoch3',
    is_trainable=False)

acc3, f1_3 = evaluate_checkpoint(
    model_eval, processor,
    llava_test, "Epoch 3")

del model_eval
torch.cuda.empty_cache()
gc.collect()
print("🧹 RAM libérée !")

# ============================================================
# Évaluation Epoch 5
# ============================================================
print("\n🔄 Chargement Epoch 5...")
model_eval = PeftModel.from_pretrained(
    model,
    '/content/drive/MyDrive/LLM/llava_rafce_epoch5',
    is_trainable=False)

acc5, f1_5 = evaluate_checkpoint(
    model_eval, processor,
    llava_test, "Epoch 5")

del model_eval
torch.cuda.empty_cache()
gc.collect()
print("🧹 RAM libérée !")

# ============================================================
# Tableau comparatif final
# ============================================================
print(f"\n{'='*55}")
print(f"📊 COMPARAISON FINALE")
print(f"{'='*55}")
print(f"{'Méthode':<25} {'Accuracy':>10} {'F1 Macro':>10}")
print(f"{'='*55}")
print(f"{'Baseline ConvNeXt':<25} {56.13:>9.2f}% {43.97:>9.2f}%")
print(f"{'LLaVA Epoch 3':<25} {acc3:>9.2f}% {f1_3:>9.2f}%")
print(f"{'LLaVA Epoch 5':<25} {acc5:>9.2f}% {f1_5:>9.2f}%")
print(f"{'='*55}")

best = max(f1_3, f1_5)
if best > 43.97:
    print(f"✅ LLaVA dépasse la Baseline !")
else:
    print(f"⚠️  LLaVA n'atteint pas encore la Baseline")

🔄 Chargement Epoch 3...

🔍 Évaluation Epoch 3


Epoch 3: 100%|██████████| 792/792 [22:55<00:00,  1.74s/it]



📊 Epoch 3 :
   Accuracy : 32.74%
   F1 Macro : 13.41%
   Erreurs  : 62/100
🧹 RAM libérée !

🔄 Chargement Epoch 5...


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(



🔍 Évaluation Epoch 5


Epoch 5: 100%|██████████| 792/792 [14:07<00:00,  1.07s/it]



📊 Epoch 5 :
   Accuracy : 30.59%
   F1 Macro : 15.12%
   Erreurs  : 7/100
🧹 RAM libérée !

📊 COMPARAISON FINALE
Méthode                     Accuracy   F1 Macro
Baseline ConvNeXt             56.13%     43.97%
LLaVA Epoch 3                 32.74%     13.41%
LLaVA Epoch 5                 30.59%     15.12%
⚠️  LLaVA n'atteint pas encore la Baseline
